In [9]:
import numpy as np
import plotly.graph_objects as go
from gym_pybullet_drones.envs.CtrlAviary import CtrlAviary
from gym_pybullet_drones.control.DSLPIDControl import DSLPIDControl
from gym_pybullet_drones.utils.enums import DroneModel, Physics

# --- 1. CONFIGURATION ---
DURATION_SEC = 10
FPS = 240
TOTAL_STEPS = DURATION_SEC * FPS
RECORD_EVERY = 10 # Record 24 frames per second (240 / 10)

# --- 2. SETUP ---
start_pos = np.array([2.0, 0.0, 1.0]) 
env = CtrlAviary(drone_model=DroneModel.CF2X, 
                 num_drones=1, 
                 initial_xyzs=np.array([start_pos]),
                 physics=Physics.PYB, 
                 gui=False)

ctrl = DSLPIDControl(drone_model=DroneModel.CF2X)

# --- 3. DEFINITIONS ---
STATIC_TARGET_POS = np.array([0.0, 0.0, 0.5]) 
LOITER_RADIUS = 1.5
LOITER_HEIGHT = 1.0
LOITER_SPEED = 1.0

# Data Storage
drone_path = []
nominal_path = [] 

action = np.zeros((1, 4))

print("Simulating...")

# --- SIMULATION LOOP ---
for i in range(TOTAL_STEPS):
    # Physics & Logic (Same as before)
    obs = env._getDroneStateVector(0)
    cur_pos = obs[0:3]
    cur_quat = obs[3:7]
    cur_vel = obs[10:13]
    cur_ang_vel = obs[13:16]
    
    t = i / FPS
    angle = LOITER_SPEED * t
    virtual_carrot = STATIC_TARGET_POS + np.array([
        LOITER_RADIUS * np.cos(angle),
        LOITER_RADIUS * np.sin(angle),
        LOITER_HEIGHT
    ])
    
    rpm, _, _ = ctrl.computeControl(control_timestep=1/FPS,
                                    cur_pos=cur_pos,
                                    cur_quat=cur_quat,
                                    cur_vel=cur_vel,
                                    cur_ang_vel=cur_ang_vel,
                                    target_pos=virtual_carrot, 
                                    target_vel=np.zeros(3))
    
    action[0, :] = rpm
    env.step(action)
    
    # RECORDING
    if i % RECORD_EVERY == 0:
        drone_path.append(cur_pos)
        nominal_path.append(virtual_carrot)

env.close()

# --- 4. OPTIMIZED ANIMATION ---
print("Generating Animation...")
drone_path = np.array(drone_path)
nominal_path = np.array(nominal_path)

fig = go.Figure()

# A. STATIC TRACES (Added once, never updated)
# Trace 0: The Tank
fig.add_trace(go.Scatter3d(
    x=[STATIC_TARGET_POS[0]], y=[STATIC_TARGET_POS[1]], z=[STATIC_TARGET_POS[2]],
    mode='markers', marker=dict(color='yellow', size=8, symbol='x'), name='Target (Static)'
))

# Trace 1: The Trail
fig.add_trace(go.Scatter3d(
    x=drone_path[:, 0], y=drone_path[:, 1], z=drone_path[:, 2],
    mode='lines', line=dict(color='white', width=2), opacity=0.3, name='Path'
))

# B. DYNAMIC TRACES (Placeholders)
# Trace 2: The Carrot
fig.add_trace(go.Scatter3d(
    x=[nominal_path[0, 0]], y=[nominal_path[0, 1]], z=[nominal_path[0, 2]],
    mode='markers', marker=dict(color='teal', size=5), name='Target Position'
))

# Trace 3: The Drone
fig.add_trace(go.Scatter3d(
    x=[drone_path[0, 0]], y=[drone_path[0, 1]], z=[drone_path[0, 2]],
    mode='markers', marker=dict(color='white', size=6), name='Drone'
))

# C. FRAMES (Only update Traces 2 and 3)
frames = []
for k in range(len(drone_path)):
    frames.append(go.Frame(
        data=[
            # Update Carrot (Trace 2)
            go.Scatter3d(x=[nominal_path[k, 0]], y=[nominal_path[k, 1]], z=[nominal_path[k, 2]]),
            # Update Drone (Trace 3)
            go.Scatter3d(x=[drone_path[k, 0]], y=[drone_path[k, 1]], z=[drone_path[k, 2]])
        ],
        traces=[2, 3], # <--- CRITICAL: Tells Plotly "Only redraw these two!"
        name=str(k)
    ))

fig.frames = frames

fig.update_layout(
    title="Loitering Mission (Optimized)",
    width=800, height=600,
    scene=dict(
        xaxis=dict(range=[-2.5, 2.5], title="X"),
        yaxis=dict(range=[-2.5, 2.5], title="Y"),
        zaxis=dict(range=[0, 2.0], title="Z"),
        aspectmode='cube'
    ),
    updatemenus=[dict(
        type="buttons", 
        buttons=[dict(label="Play", 
                      method="animate", 
                      args=[None, dict(frame=dict(duration=20, redraw=True), fromcurrent=True)])]
    )]
)

fig.show()

[INFO] BaseAviary.__init__() loaded parameters from the drone's .urdf:
[INFO] m 0.027000, L 0.039700,
[INFO] ixx 0.000014, iyy 0.000014, izz 0.000022,
[INFO] kf 3.160000e-10, km 7.940000e-12,
[INFO] t2w 2.250000, max_speed_kmh 30.000000,
[INFO] gnd_eff_coeff 11.368590, prop_radius 0.023135,
[INFO] drag_xy_coeff 0.000001, drag_z_coeff 0.000001,
[INFO] dw_coeff_1 2267.180000, dw_coeff_2 0.160000, dw_coeff_3 -0.110000
Simulating...
Generating Animation...
